# Retrieval v2 – Stage 0 / Stage 1 / Stage 2 / Stage 3

This notebook is a thin control and inspection layer.

- **Stage 0** reproduces the frozen Baseline v1 retrieval behavior.
- **Stage 1** evaluates lexical retrieval and freezes `L2_bm25_preserving`.
- **Stage 2** evaluates dense retrieval and freezes `D1_e5_large_instruct`.
- **Stage 3** reuses the frozen L2 and D1 window-score caches and compares video aggregation policies without rerunning retrievers or neural models.
- Retrieval algorithms and metrics live in `src/`, not in notebook cells.
- Holdout20 remains closed.


## 1. Setup

Mount Google Drive, resolve the code root and artifact root, and verify the Retrieval v2 files.


In [1]:
from pathlib import Path
import glob
import json
import os
import subprocess
import sys

import pandas as pd
from IPython.display import display

try :
    from google.colab import drive
    drive.mount("/content/drive", force_remount = False)
except Exception :
    pass


CODE_ROOT_OVERRIDE     = ""
ARTIFACT_ROOT_OVERRIDE = ""


def first_existing(candidates) :
    for candidate in candidates :
        path = Path(candidate)

        if (path.exists()) :
            return path.resolve()

    return None


def detect_artifact_root() :
    if (ARTIFACT_ROOT_OVERRIDE) :
        return Path(ARTIFACT_ROOT_OVERRIDE).expanduser().resolve()

    environment = os.environ.get("AIC_RETRIEVAL_ARTIFACT_ROOT", "").strip()

    if (environment) :
        return Path(environment).expanduser().resolve()

    candidates = ["/content/drive/MyDrive/aic26/asr_model_comparison"]
    candidates.extend(
        glob.glob(
            "/content/drive/.shortcut-targets-by-id/*/aic26/asr_model_comparison"
        )
    )
    result = first_existing(candidates)

    if (result is None) :
        raise FileNotFoundError(
            "Could not detect the ASR artifact workspace. "
            "Set ARTIFACT_ROOT_OVERRIDE in this cell."
        )

    return result


def detect_code_root(artifact_root) :
    if (CODE_ROOT_OVERRIDE) :
        return Path(CODE_ROOT_OVERRIDE).expanduser().resolve()

    environment = os.environ.get("AIC_RETRIEVAL_CODE_ROOT", "").strip()

    if (environment) :
        return Path(environment).expanduser().resolve()

    candidates = [
        Path.cwd(),
        Path("/content/aic-2026-asr-retrieval"),
        artifact_root,
    ]

    for candidate in candidates :
        if ((candidate / "configs" / "retrieval_v2.json").exists()) :
            return candidate.resolve()

    raise FileNotFoundError(
        "Could not detect the Retrieval v2 code root. "
        "Clone/copy the repository first or set CODE_ROOT_OVERRIDE."
    )


ARTIFACT_ROOT = detect_artifact_root()
CODE_ROOT     = detect_code_root(ARTIFACT_ROOT)

os.environ["AIC_RETRIEVAL_ARTIFACT_ROOT"] = str(ARTIFACT_ROOT)
os.environ["AIC_RETRIEVAL_CODE_ROOT"]     = str(CODE_ROOT)

RUNNER = CODE_ROOT / "src" / "07_evaluate_retrieval_v2.py"
CONFIG = CODE_ROOT / "configs" / "retrieval_v2.json"

print("Code root    :", CODE_ROOT)
print("Artifact root:", ARTIFACT_ROOT)
print("Runner exists:", RUNNER.exists())
print("Config exists:", CONFIG.exists())

assert RUNNER.exists()
assert CONFIG.exists()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Code root    : /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Artifact root: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Runner exists: True
Config exists: True


## 2. Stage selection

Stage 3 requires the already-passed Stage 0, Stage 1, and Stage 2 contracts. It consumes only the frozen L2 BM25 and D1 E5-large score caches.


In [2]:
STAGE = "stage03_aggregation"

assert STAGE in {
    "stage00_baseline",
    "stage01_lexical",
    "stage02_dense",
    "stage03_aggregation",
}
print("Selected stage:", STAGE)

Selected stage: stage03_aggregation


## 3. Configuration and integrity preview

Inspect the frozen experiment contract before execution.


In [3]:
config = json.loads(CONFIG.read_text(encoding = "utf-8"))

print("Retrieval ID :", config["retrieval_id"])
print("Frozen       :", config["retrieval_frozen"])

print("\nChannels:")
for channel_id, item in config["channels"].items() :
    print(f"  {channel_id:20s} model={item['model_id']:32s} view={item['view']}")

print("\nEvaluation policy:")
print(json.dumps(config["evaluation_policy"], ensure_ascii = False, indent = 2))

if (STAGE == "stage00_baseline") :
    print("\nStage 0 semantic model:")
    print(json.dumps(config["stage00_baseline"]["semantic"], ensure_ascii = False, indent = 2))
elif (STAGE == "stage01_lexical") :
    print("\nStage 1 lexical methods:")
    print(json.dumps(config["stage01_lexical"], ensure_ascii = False, indent = 2))
elif (STAGE == "stage02_dense") :
    print("\nStage 2 dense backends:")
    for backend_id, item in config["stage02_dense"]["backends"].items() :
        print(f"  {backend_id:28s} {item['model_name']}@{item['revision'][:12]}")
else :
    stage3 = config["stage03_aggregation"]
    print("\nStage 3 frozen score sources:")
    for source_id, item in stage3["sources"].items() :
        expected_scores = item.get("expected_window_scores_sha256") or "<from source stage_summary>"
        expected_axes   = item.get("expected_score_axes_sha256") or "<from source stage_summary>"
        print(f"  {source_id:26s} {item['source_stage']} / {item['method_id']}")
        print("    score hash:", expected_scores)
        print("    axes hash :", expected_axes)

    print("\nStage 3 aggregation methods:")
    for method_id, item in stage3["methods"].items() :
        print(f"  {method_id:24s} {item}")

print("\nStrict Stage 0 fixture:")
print(json.dumps(config["regression"], ensure_ascii = False, indent = 2))

Retrieval ID : aic2026_retrieval_v2_stage00_stage03
Frozen       : False

Channels:
  whisper_raw          model=whisper_large_v3                 view=raw
  whisper_processed    model=whisper_large_v3                 view=processed
  parakeet_raw         model=parakeet_ctc_0_6b_vietnamese     view=raw
  parakeet_processed   model=parakeet_ctc_0_6b_vietnamese     view=processed

Evaluation policy:
{
  "silver_radius_s": 60.0,
  "minimum_overlap_s": 30.0,
  "video_aggregation": "max_window",
  "tie_tolerance": 1e-12,
  "tie_policy": "worst_rank_within_tolerance",
  "story_scope": "correct_video_only"
}

Stage 3 frozen score sources:
  S0_bm25_preserving         stage01_lexical / L2_bm25_preserving
    score hash: <from source stage_summary>
    axes hash : <from source stage_summary>
  S1_e5_large_instruct       stage02_dense / D1_e5_large_instruct
    score hash: <from source stage_summary>
    axes hash : <from source stage_summary>

Stage 3 aggregation methods:
  P0_max               

In [4]:
import json
from pathlib import Path
import numpy as np

ROOT = Path("/content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison")

core = json.loads((ROOT / "data/benchmark_manifest.json").read_text())
ext  = json.loads((ROOT / "data/stage1_extension40/benchmark_manifest.json").read_text())

records = []

for benchmark in [core, ext] :
    windows = benchmark["windows"]

    for video_id in sorted({str(x["video_id"]) for x in windows}) :
        records.extend([
            x for x in windows
            if str(x["video_id"]) == video_id
        ])

expected = [str(x["window_id"]) for x in records]

axes = json.loads(
    (ROOT / "cache/retrieval_v2/stage01_lexical/score_axes.json").read_text()
)

cached = next(
    x["window_ids"]
    for x in axes["bundles"]
    if x["query_set"] == "development20"
    and x["method_id"] == "L2_bm25_preserving"
    and x["model_id"] == "whisper_large_v3"
    and x["view"] == "raw"
)

print("Expected windows:", len(expected))
print("Cached windows:  ", len(cached))
print("Exact axis match:", expected == cached)

Expected windows: 994
Cached windows:   994
Exact axis match: True


## 4. Execute Retrieval v2

The runner streams live output. Stage 3 should not load a neural backend or recompute BM25/E5 scores.


In [5]:
command = [sys.executable, "-u", str(RUNNER), "--stage", STAGE]
env     = {**os.environ, "PYTHONUNBUFFERED" : "1"}

print("Running:", " ".join(command), "\n")
process = subprocess.Popen(
    command,
    cwd = str(CODE_ROOT),
    env = env,
    stdout = subprocess.PIPE,
    stderr = subprocess.STDOUT,
    text = True,
    bufsize = 1,
)
for line in process.stdout :
    print(line, end = "", flush = True)

returncode = process.wait()

if (returncode != 0) :
    raise RuntimeError(f"Retrieval v2 failed with exit code {returncode}")


Running: /usr/bin/python3 -u /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/src/07_evaluate_retrieval_v2.py --stage stage03_aggregation 

AIC 2026 RETRIEVAL V2
Stage:         stage03_aggregation
Code root:     /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Artifact root: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison
Reports:       /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/reports/retrieval_v2/stage03_aggregation

[1/6] Validating Stage 3 contract and benchmark identities...
      done in 0.1s
[2/6] Validating development20, All50, and prerequisites...
      done in 0.0s
[3/6] Loading frozen L2/D1 window-score caches and aggregating videos...
      done in 114.3s
[4/6] Checking P0 reproduction, story invariance, and holdout exclusion...
      done in 0.0s
[5/6] Writing Sta

## 5. Stage result

Load the machine-readable summary and validation gates.


In [6]:
REPORT_ROOT = ARTIFACT_ROOT / config["paths"]["reports_root"] / STAGE

summary_path    = REPORT_ROOT / "stage_summary.json"
validation_path = REPORT_ROOT / "validation_summary.json"

summary    = json.loads(summary_path.read_text(encoding = "utf-8"))
validation = json.loads(validation_path.read_text(encoding = "utf-8"))

print("=" * 88)
print("STAGE RESULT")
print("=" * 88)
print("Stage :", summary["stage"])
print("Status:", "PASS" if summary["passed"] else "FAIL")
print("Reports:", REPORT_ROOT)
print("Validation:", "PASS" if validation["passed"] else "FAIL")

if (STAGE == "stage03_aggregation") :
    print("\nStage 3 gates:")
    for key in [
        "p0_regression_passed",
        "story_invariance_passed",
        "temporal_axis_validation_passed",
        "source_validation_passed",
    ] :
        print(f"  {key:34s}: {summary.get(key)}")


STAGE RESULT
Stage : stage03_aggregation
Status: PASS
Reports: /content/drive/.shortcut-targets-by-id/1VXCKOD57uox_lUy-66Vqz5cKfzk8cvs4/aic26/asr_model_comparison/reports/retrieval_v2/stage03_aggregation
Validation: PASS

Stage 3 gates:
  p0_regression_passed              : True
  story_invariance_passed           : True
  temporal_axis_validation_passed   : True
  source_validation_passed          : True


## 6. Aggregate metrics

Stage 3 metrics are grouped by frozen source, aggregation policy, ASR, and transcript view.


In [7]:
metrics = pd.read_csv(REPORT_ROOT / "retrieval_metrics.csv")

if (STAGE == "stage03_aggregation") :
    metric_columns = [
        "source_id", "source_method_id", "aggregation_id",
        "model_id", "view", "query_count",
        "story_recall_at_1", "story_mrr",
        "video_recall_at_1", "video_recall_at_3",
        "video_recall_at_5", "video_recall_at_10",
        "video_recall_at_20", "video_mrr",
    ]
    display(
        metrics[[column for column in metric_columns if column in metrics.columns]]
        .sort_values(["source_id", "aggregation_id", "model_id", "view"])
        .reset_index(drop = True)
    )
else :
    metric_columns = [
        "query_set", "method_id", "model_id", "view", "query_count",
        "story_recall_at_1", "story_recall_at_3", "story_recall_at_5",
        "story_recall_at_10", "story_mrr", "video_recall_at_1",
        "video_recall_at_3", "video_recall_at_5", "video_recall_at_10",
        "video_recall_at_20", "video_mrr",
    ]
    display(
        metrics[[column for column in metric_columns if column in metrics.columns]]
        .sort_values(["query_set", "method_id", "model_id", "view"])
        .reset_index(drop = True)
    )


,source_id,source_method_id,aggregation_id,model_id,view,query_count,story_recall_at_1,story_mrr,video_recall_at_1,video_recall_at_3,video_recall_at_5,video_recall_at_10,video_recall_at_20,video_mrr
0,S0_bm25_preserving,L2_bm25_preserving,P0_max,parakeet_ctc_0_6b_vietnamese,processed,20,0.75,0.795989,0.45,0.65,0.65,0.80,0.85,0.571989
1,S0_bm25_preserving,L2_bm25_preserving,P0_max,parakeet_ctc_0_6b_vietnamese,raw,20,0.75,0.795989,0.45,0.65,0.65,0.80,0.85,0.571989
2,S0_bm25_preserving,L2_bm25_preserving,P0_max,whisper_large_v3,processed,20,0.70,0.776234,0.50,0.70,0.80,0.90,0.90,0.622708
3,S0_bm25_preserving,L2_bm25_preserving,P0_max,whisper_large_v3,raw,20,0.70,0.767522,0.55,0.75,0.85,0.90,0.90,0.658864
4,S0_bm25_preserving,L2_bm25_preserving,P1_top2_mean,parakeet_ctc_0_6b_vietnamese,processed,20,0.75,0.795989,0.45,0.55,0.70,0.80,0.90,0.548975
5,S0_bm25_preserving,L2_bm25_preserving,P1_top2_mean,parakeet_ctc_0_6b_vietnamese,raw,20,0.75,0.795989,0.45,0.55,0.70,0.80,0.90,0.548975
6,S0_bm25_preserving,L2_bm25_preserving,P1_top2_mean,whisper_large_v3,processed,20,0.70,0.776234,0.55,0.70,0.85,0.90,0.90,0.659387
7,S0_bm25_preserving,L2_bm25_preserving,P1_top2_mean,whisper_large_v3,raw,20,0.70,0.767522,0.55,0.70,0.80,0.85,0.90,0.654697
8,S0_bm25_preserving,L2_bm25_preserving,P2_top3_mean,parakeet_ctc_0_6b_vietnamese,processed,20,0.75,0.795989,0.40,0.60,0.65,0.80,0.85,0.521860
9,S0_bm25_preserving,L2_bm25_preserving,P2_top3_mean,parakeet_ctc_0_6b_vietnamese,raw,20,0.75,0.795989,0.40,0.60,0.65,0.80,0.85,0.521860


## 7. Paired comparison

For Stage 3, inspect P1–P4 against P0 separately for BM25 and E5-large before considering the cross-source summary.


In [8]:
query_results = pd.read_csv(REPORT_ROOT / "query_results.csv")

if (STAGE == "stage03_aggregation") :
    method_summary   = pd.read_csv(REPORT_ROOT / "method_summary.csv")
    query_comparison = pd.read_csv(REPORT_ROOT / "query_comparison.csv")

    print("Stage 3 cross-source method summary:")
    display(method_summary)

    print("\nLargest improvements vs P0:")
    display(
        query_comparison.sort_values(
            ["video_rr_delta", "candidate_video_rank"],
            ascending = [False, True],
        ).head(40)
    )

    print("\nLargest regressions vs P0:")
    display(
        query_comparison.sort_values(
            ["video_rr_delta", "candidate_video_rank"],
            ascending = [True, False],
        ).head(40)
    )
elif (STAGE == "stage01_lexical") :
    display(pd.read_csv(REPORT_ROOT / "method_summary.csv"))
    display(pd.read_csv(REPORT_ROOT / "query_comparison.csv").head(80))
elif (STAGE == "stage02_dense") :
    display(pd.read_csv(REPORT_ROOT / "method_summary.csv"))
    display(pd.read_csv(REPORT_ROOT / "query_comparison.csv").head(120))
else :
    columns = [
        "query_set", "model_id", "view", "query_id",
        "first_relevant_rank", "video_rank",
        "top_story_window_id", "top_video_id",
        "story_score_margin", "video_score_margin",
    ]
    display(query_results[columns].head(120))


Stage 3 cross-source method summary:


,aggregation_id,bm25_video_rr_mean,bm25_delta_mean_vs_P0,bm25_delta_median_vs_P0,bm25_better,bm25_tie,bm25_worse,bm25_bootstrap_90_low,bm25_bootstrap_90_high,dense_video_rr_mean,...,dense_bootstrap_90_low,dense_bootstrap_90_high,cross_source_video_rr_mean,cross_source_delta_mean_vs_P0,cross_source_delta_median_vs_P0,cross_source_better,cross_source_tie,cross_source_worse,cross_source_bootstrap_90_low,cross_source_bootstrap_90_high
0,P0_max,0.606387,0.000000,0.0,0,20,0,0.000000,0.000000,0.730139,...,0.000000,0.000000,0.668263,0.000000,0.0,0,20,0,0.000000,0.000000
1,P1_top2_mean,0.603009,-0.003379,0.0,8,9,3,-0.064841,0.047303,0.744319,...,-0.044708,0.067203,0.673664,0.005400,0.0,9,7,4,-0.039968,0.049016
2,P2_top3_mean,0.564737,-0.041650,0.0,7,7,6,-0.107340,0.019967,0.677635,...,-0.157386,0.038316,0.621186,-0.047078,0.0,8,5,7,-0.114650,0.014089
3,P3_adjacent2_mean,0.603195,-0.003192,0.0,7,8,5,-0.066239,0.049637,0.710734,...,-0.087769,0.038511,0.656964,-0.011299,0.0,8,6,6,-0.054983,0.027228
4,P4_contiguous3_mean,0.574894,-0.031493,0.0,7,6,7,-0.125566,0.057197,0.664749,...,-0.170452,0.030112,0.619821,-0.048442,0.0,9,4,7,-0.123924,0.022491



Largest improvements vs P0:


,source_id,source_method_id,aggregation_id,reference_aggregation_id,model_id,view,query_id,reference_video_rank,candidate_video_rank,reference_video_rr,candidate_video_rr,video_rank_delta,video_rr_delta,outcome,top_video_changed
502,S0_bm25_preserving,L2_bm25_preserving,P4_contiguous3_mean,P0_max,whisper_large_v3,processed,R1-13,4,1,0.250000,1.0,-3,0.750000,better,True
1,S0_bm25_preserving,L2_bm25_preserving,P1_top2_mean,P0_max,whisper_large_v3,raw,R1-9,3,1,0.333333,1.0,-2,0.666667,better,True
21,S0_bm25_preserving,L2_bm25_preserving,P1_top2_mean,P0_max,whisper_large_v3,processed,R1-9,3,1,0.333333,1.0,-2,0.666667,better,True
98,S1_e5_large_instruct,D1_e5_large_instruct,P1_top2_mean,P0_max,whisper_large_v3,raw,R3-20,3,1,0.333333,1.0,-2,0.666667,better,True
143,S1_e5_large_instruct,D1_e5_large_instruct,P1_top2_mean,P0_max,parakeet_ctc_0_6b_vietnamese,processed,R1-15,3,1,0.333333,1.0,-2,0.666667,better,True
161,S0_bm25_preserving,L2_bm25_preserving,P2_top3_mean,P0_max,whisper_large_v3,raw,R1-9,3,1,0.333333,1.0,-2,0.666667,better,True
181,S0_bm25_preserving,L2_bm25_preserving,P2_top3_mean,P0_max,whisper_large_v3,processed,R1-9,3,1,0.333333,1.0,-2,0.666667,better,True
258,S1_e5_large_instruct,D1_e5_large_instruct,P2_top3_mean,P0_max,whisper_large_v3,raw,R3-20,3,1,0.333333,1.0,-2,0.666667,better,True
303,S1_e5_large_instruct,D1_e5_large_instruct,P2_top3_mean,P0_max,parakeet_ctc_0_6b_vietnamese,processed,R1-15,3,1,0.333333,1.0,-2,0.666667,better,True
321,S0_bm25_preserving,L2_bm25_preserving,P3_adjacent2_mean,P0_max,whisper_large_v3,raw,R1-9,3,1,0.333333,1.0,-2,0.666667,better,True



Largest regressions vs P0:


,source_id,source_method_id,aggregation_id,reference_aggregation_id,model_id,view,query_id,reference_video_rank,candidate_video_rank,reference_video_rr,candidate_video_rr,video_rank_delta,video_rr_delta,outcome,top_video_changed
287,S1_e5_large_instruct,D1_e5_large_instruct,P2_top3_mean,P0_max,parakeet_ctc_0_6b_vietnamese,raw,R2-11,1,42,1.0,0.023810,41,-0.976190,worse,True
607,S1_e5_large_instruct,D1_e5_large_instruct,P4_contiguous3_mean,P0_max,parakeet_ctc_0_6b_vietnamese,raw,R2-11,1,40,1.0,0.025000,39,-0.975000,worse,True
247,S1_e5_large_instruct,D1_e5_large_instruct,P2_top3_mean,P0_max,whisper_large_v3,raw,R2-11,1,37,1.0,0.027027,36,-0.972973,worse,True
574,S1_e5_large_instruct,D1_e5_large_instruct,P4_contiguous3_mean,P0_max,whisper_large_v3,raw,R2-20,1,25,1.0,0.040000,24,-0.960000,worse,True
567,S1_e5_large_instruct,D1_e5_large_instruct,P4_contiguous3_mean,P0_max,whisper_large_v3,raw,R2-11,1,24,1.0,0.041667,23,-0.958333,worse,True
414,S1_e5_large_instruct,D1_e5_large_instruct,P3_adjacent2_mean,P0_max,whisper_large_v3,raw,R2-20,1,22,1.0,0.045455,21,-0.954545,worse,True
267,S1_e5_large_instruct,D1_e5_large_instruct,P2_top3_mean,P0_max,whisper_large_v3,processed,R2-11,1,19,1.0,0.052632,18,-0.947368,worse,True
314,S1_e5_large_instruct,D1_e5_large_instruct,P2_top3_mean,P0_max,parakeet_ctc_0_6b_vietnamese,processed,R2-20,1,10,1.0,0.100000,9,-0.900000,worse,True
587,S1_e5_large_instruct,D1_e5_large_instruct,P4_contiguous3_mean,P0_max,whisper_large_v3,processed,R2-11,1,10,1.0,0.100000,9,-0.900000,worse,True
634,S1_e5_large_instruct,D1_e5_large_instruct,P4_contiguous3_mean,P0_max,parakeet_ctc_0_6b_vietnamese,processed,R2-20,1,8,1.0,0.125000,7,-0.875000,worse,True


## 8. Aggregation diagnostics

Inspect how the correct video and strongest wrong video differ in peak isolation and selected support windows.


In [9]:
if (STAGE == "stage03_aggregation") :
    diagnostics = pd.read_csv(REPORT_ROOT / "aggregation_diagnostics.csv")
    QUERY_ID = "R2-3" if "R2-3" in set(diagnostics["query_id"]) else diagnostics["query_id"].iloc[0]

    print("Diagnostic query:", QUERY_ID)
    columns = [
        "source_id", "aggregation_id", "model_id", "view", "query_id",
        "correct_video_id", "correct_video_rank",
        "correct_video_peak_score", "correct_video_isolation_gap",
        "correct_video_support_window_ids",
        "best_wrong_video_id", "best_wrong_video_peak_score",
        "best_wrong_video_isolation_gap",
        "best_wrong_video_support_window_ids",
        "margin",
    ]
    display(
        diagnostics[diagnostics["query_id"] == QUERY_ID]
        [[column for column in columns if column in diagnostics.columns]]
        .sort_values(["source_id", "aggregation_id", "model_id", "view"])
        .reset_index(drop = True)
    )
elif (STAGE in {"stage01_lexical", "stage02_dense"}) :
    evidence = pd.read_csv(REPORT_ROOT / "top_evidence.csv")
    QUERY_ID = "R2-20" if "R2-20" in set(evidence["query_id"]) else evidence["query_id"].iloc[0]
    display(evidence[evidence["query_id"] == QUERY_ID].head(160))


Diagnostic query: R2-3


,source_id,aggregation_id,model_id,view,query_id,correct_video_id,correct_video_rank,correct_video_peak_score,correct_video_isolation_gap,correct_video_support_window_ids,best_wrong_video_id,best_wrong_video_peak_score,best_wrong_video_isolation_gap,best_wrong_video_support_window_ids,margin
0,S0_bm25_preserving,P0_max,parakeet_ctc_0_6b_vietnamese,processed,R2-3,K17_V003,6,42.185402,5.405708,K17_V003_0008,K12_V001,51.938103,11.307709,K12_V001_0016,-9.752701
1,S0_bm25_preserving,P0_max,parakeet_ctc_0_6b_vietnamese,raw,R2-3,K17_V003,6,42.191883,5.405632,K17_V003_0008,K12_V001,51.946602,11.309669,K12_V001_0016,-9.754719
2,S0_bm25_preserving,P0_max,whisper_large_v3,processed,R2-3,K17_V003,5,42.712082,6.747856,K17_V003_0008,K12_V001,52.899925,12.587791,K12_V001_0016,-10.187843
3,S0_bm25_preserving,P0_max,whisper_large_v3,raw,R2-3,K17_V003,5,42.982044,6.825562,K17_V003_0008,K12_V001,53.117023,12.644608,K12_V001_0016,-10.134979
4,S0_bm25_preserving,P1_top2_mean,parakeet_ctc_0_6b_vietnamese,processed,R2-3,K17_V003,5,42.185402,5.405708,K17_V003_0008|K17_V003_0007,K12_V001,51.938103,11.307709,K12_V001_0016|K12_V001_0017,-6.801701
5,S0_bm25_preserving,P1_top2_mean,parakeet_ctc_0_6b_vietnamese,raw,R2-3,K17_V003,5,42.191883,5.405632,K17_V003_0008|K17_V003_0007,K12_V001,51.946602,11.309669,K12_V001_0016|K12_V001_0017,-6.802700
6,S0_bm25_preserving,P1_top2_mean,whisper_large_v3,processed,R2-3,K17_V003,5,42.712082,6.747856,K17_V003_0008|K17_V003_0021,K12_V001,52.899925,12.587791,K12_V001_0016|K12_V001_0017,-7.076496
7,S0_bm25_preserving,P1_top2_mean,whisper_large_v3,raw,R2-3,K17_V003,5,42.982044,6.825562,K17_V003_0008|K17_V003_0021,K12_V001,53.117023,12.644608,K12_V001_0016|K12_V001_0017,-7.007570
8,S0_bm25_preserving,P2_top3_mean,parakeet_ctc_0_6b_vietnamese,processed,R2-3,K17_V003,4,42.185402,5.405708,K17_V003_0008|K17_V003_0007|K17_V003_0009,K05_V013,50.482498,13.193733,K05_V013_0021|K05_V013_0001|K05_V013_0020,-3.565272
9,S0_bm25_preserving,P2_top3_mean,parakeet_ctc_0_6b_vietnamese,raw,R2-3,K17_V003,4,42.191883,5.405632,K17_V003_0008|K17_V003_0007|K17_V003_0009,K05_V013,50.490734,13.195930,K05_V013_0021|K05_V013_0001|K05_V013_0020,-3.566219


## 9. Aggregation latency

This is **incremental cached-score video aggregation only**, not full query retrieval latency.


In [10]:
if (STAGE == "stage03_aggregation") :
    latency = pd.read_csv(REPORT_ROOT / "latency_summary.csv")
    display(
        latency.sort_values(
            ["source_id", "aggregation_id", "model_id", "view"]
        ).reset_index(drop = True)
    )


,source_id,aggregation_id,model_id,view,query_count,aggregation_mean_ms,aggregation_p50_ms,aggregation_p90_ms
0,S0_bm25_preserving,P0_max,parakeet_ctc_0_6b_vietnamese,processed,20,134.917565,134.569665,154.574472
1,S0_bm25_preserving,P0_max,parakeet_ctc_0_6b_vietnamese,raw,20,143.276411,142.371886,155.824453
2,S0_bm25_preserving,P0_max,whisper_large_v3,processed,20,131.112874,130.248419,159.182739
3,S0_bm25_preserving,P0_max,whisper_large_v3,raw,20,114.055466,111.727534,121.220144
4,S0_bm25_preserving,P1_top2_mean,parakeet_ctc_0_6b_vietnamese,processed,20,147.481951,142.985949,169.791422
5,S0_bm25_preserving,P1_top2_mean,parakeet_ctc_0_6b_vietnamese,raw,20,125.375069,112.963493,168.326092
6,S0_bm25_preserving,P1_top2_mean,whisper_large_v3,processed,20,111.196750,108.735183,120.478321
7,S0_bm25_preserving,P1_top2_mean,whisper_large_v3,raw,20,112.635803,109.772383,124.692470
8,S0_bm25_preserving,P2_top3_mean,parakeet_ctc_0_6b_vietnamese,processed,20,108.828529,105.836983,117.091870
9,S0_bm25_preserving,P2_top3_mean,parakeet_ctc_0_6b_vietnamese,raw,20,110.429856,108.763843,117.864760


## 10. Research decision

Do not select an aggregation automatically. Review quality, paired behavior, cross-source consistency, failure recovery, and simplicity.


In [11]:
RETRIEVAL_DECISION = {
    "stage" : STAGE,
    "lexical_default" : "L2_bm25_preserving",
    "dense_default" : "D1_e5_large_instruct",
    "selected_video_aggregation" : None,
    "retained_aggregation_alternative" : None,
    "main_observation" : None,
    "failure_observation" : None,
    "notes" : [
        "Stage 3 compares P1-P4 against P0 max using frozen BM25 and E5-large scores.",
        "Story retrieval must remain invariant because Stage 3 changes only video aggregation.",
        "Whisper and Parakeet remain independent diagnostic conditions.",
        "Raw and processed remain independent transcript views.",
        "Sparse+dense fusion is deferred to Stage 4.",
        "Holdout20 remains closed.",
    ],
}

print(json.dumps(RETRIEVAL_DECISION, ensure_ascii = False, indent = 2))


{
  "stage": "stage03_aggregation",
  "lexical_default": "L2_bm25_preserving",
  "dense_default": "D1_e5_large_instruct",
  "selected_video_aggregation": null,
  "retained_aggregation_alternative": null,
  "main_observation": null,
  "failure_observation": null,
  "notes": [
    "Stage 3 compares P1-P4 against P0 max using frozen BM25 and E5-large scores.",
    "Story retrieval must remain invariant because Stage 3 changes only video aggregation.",
    "Whisper and Parakeet remain independent diagnostic conditions.",
    "Raw and processed remain independent transcript views.",
    "Sparse+dense fusion is deferred to Stage 4.",
    "Holdout20 remains closed."
  ]
}
